In [ ]:
!nvidia-smi
![ -d "SE374.R11-Group11" ] && (cd SE374.R11-Group11 && git pull) || git clone https://github.com/EmmYeuuKhoaaHocc/SE374.R11-Group11.git
%cd /content/SE374.R11-Group11/apps/worker

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass

# Enter your ngrok TCP endpoint exposed for local RabbitMQ port 5672 (e.g. 0.tcp.ap.ngrok.io and 12345)
rmq_host = input("RabbitMQ ngrok host (e.g. 0.tcp.ap.ngrok.io): ").strip()
rmq_port = input("RabbitMQ ngrok TCP port: ").strip()
rmq_user = input("RabbitMQ username: ").strip()
rmq_pass = getpass("RabbitMQ password: ")
print(f"Configured RabbitMQ target: {rmq_host}:{rmq_port} as {rmq_user}")

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["RABBITMQ_USER"] = rmq_user
env["RABBITMQ_PASS"] = rmq_pass

subprocess.run(["fuser", "-k", "8000/tcp"], stderr=subprocess.DEVNULL)

log_path = "/content/SE374.R11-Group11/apps/worker/worker.log"
with open(log_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            "python", "-u", "main.py",
            "--host", "0.0.0.0",
            "--port", "8000",
            "--mc-batch-size", "5",
            "--rmq-host", rmq_host,
            "--rmq-port", str(rmq_port),
        ],
        cwd="/content/SE374.R11-Group11/apps/worker",
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

health_data = None
for _ in range(30):
    if proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            health_data = json.loads(resp.read().decode())
            if not health_data.get("rabbitmq_enabled") or health_data.get("rabbitmq_connected"):
                break
    except Exception:
        pass
    time.sleep(1)

with open(log_path) as f:
    print(f.read())

if health_data is None:
    raise RuntimeError(f"Worker did not become healthy (exit_code={proc.poll()}). Check worker.log above.")

print("Health check:", json.dumps(health_data, indent=2))
if health_data.get("rabbitmq_enabled") and not health_data.get("rabbitmq_connected"):
    raise RuntimeError(f"Worker is up, but RabbitMQ connection failed: {health_data.get('rabbitmq_error')}")

In [ ]:
!tail -n 60 /content/SE374.R11-Group11/apps/worker/worker.log